In [10]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
import pandas as pd

In [4]:
print(spark.version)

3.5.5


In [11]:
#to read csv
# df=spark.read.csv('./data/raw/online_retail_II.xlsx',header=True, inferSchema=True)

pandas_df = pd.read_excel('online_retail_II.xlsx')

AttributeError: 'SparkSession' object has no attribute 'createDataFreame'

In [12]:
df=spark.createDataFrame(pandas_df ) 
#df.show(3)
df.show(1,truncate = False)

26/04/19 11:56:44 WARN TaskSetManager: Stage 0 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|Customer ID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085.0    |United Kingdom|
|489434 |79323P   |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75 |13085.0    |United Kingdom|
|489434 |79323W   | WHITE CHERRY LIGHTS               |12      |2009-12-01 07:45:00|6.75 |13085.0    |United Kingdom|
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
only showing top 3 rows



In [13]:
df.show(1,truncate = False)

+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|Customer ID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085.0    |United Kingdom|
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
only showing top 1 row



26/04/19 11:58:04 WARN TaskSetManager: Stage 1 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


In [14]:
df.printSchema()# shows column names and data types
df.count() # number of rows


root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



26/04/19 12:05:37 WARN TaskSetManager: Stage 2 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

8

In [15]:
len(df.columns) # number of columns

8

In [16]:
df.count() # number of rows

26/04/19 12:06:43 WARN TaskSetManager: Stage 5 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


525461

In [17]:
from pyspark.sql.functions import col,sum as spark_sum
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

26/04/19 12:09:43 WARN TaskSetManager: Stage 8 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|      0|        0|          0|       0|          0|    0|          0|      0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+



In [27]:
#df.select('Country','Price').show()
from pyspark.sql.functions import max, min, avg, sum
df.select(
    max("Price"),
    min("Price"),
    avg("Price"),
    sum("Price")).show()


26/04/19 12:16:26 WARN TaskSetManager: Stage 15 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


+----------+----------+-----------------+-----------------+
|max(Price)|min(Price)|       avg(Price)|       sum(Price)|
+----------+----------+-----------------+-----------------+
|  25111.09| -53594.36|4.688834478677021|2463799.654000106|
+----------+----------+-----------------+-----------------+



In [29]:
df.filter(df['Price']== -53594.36).show()

26/04/19 12:21:17 WARN TaskSetManager: Stage 18 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 12:21:17 WARN TaskSetManager: Stage 19 contains a task of very large size (2082 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 12:21:17 WARN TaskSetManager: Stage 20 contains a task of very large size (2033 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+
|Invoice|StockCode|    Description|Quantity|        InvoiceDate|    Price|Customer ID|       Country|
+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+
|A506401|        B|Adjust bad debt|       1|2010-04-29 13:36:00|-53594.36|        NaN|United Kingdom|
+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+



In [30]:
df.filter(df['Price']<0).show()

26/04/19 12:22:35 WARN TaskSetManager: Stage 21 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 12:22:35 WARN TaskSetManager: Stage 22 contains a task of very large size (2082 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 12:22:35 WARN TaskSetManager: Stage 23 contains a task of very large size (2033 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+
|Invoice|StockCode|    Description|Quantity|        InvoiceDate|    Price|Customer ID|       Country|
+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+
|A506401|        B|Adjust bad debt|       1|2010-04-29 13:36:00|-53594.36|        NaN|United Kingdom|
|A516228|        B|Adjust bad debt|       1|2010-07-19 11:24:00|-44031.79|        NaN|United Kingdom|
|A528059|        B|Adjust bad debt|       1|2010-10-20 12:04:00|-38925.87|        NaN|United Kingdom|
+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+



In [34]:

df.groupBy('Description').count().orderBy("count", ascending= False).show()
#df.groupBy("Description").count().orderBy("count", ascending=False).show(10)

26/04/19 12:27:10 WARN TaskSetManager: Stage 24 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
[Stage 24:>                                                       (0 + 16) / 16]

+--------------------+-----+
|         Description|count|
+--------------------+-----+
|WHITE HANGING HEA...| 3549|
|                 NaN| 2928|
|REGENCY CAKESTAND...| 2212|
|STRAWBERRY CERAMI...| 1843|
|PACK OF 72 RETRO ...| 1466|
|ASSORTED COLOUR B...| 1457|
|60 TEATIME FAIRY ...| 1400|
|HOME BUILDING BLO...| 1386|
|JUMBO BAG RED RET...| 1310|
|LUNCH BAG RED SPOTTY| 1274|
|REX CASH+CARRY JU...| 1232|
|JUMBO STORAGE BAG...| 1220|
|PACK OF 60 PINK P...| 1196|
|WOODEN FRAME ANTI...| 1190|
|LUNCH BAG  BLACK ...| 1179|
|BAKING SET 9 PIEC...| 1176|
|LUNCH BAG SUKI  D...| 1157|
|HEART OF WICKER L...| 1151|
|LOVE BUILDING BLO...| 1142|
|RED HANGING HEART...| 1129|
+--------------------+-----+
only showing top 20 rows



In [36]:
df.filter(df['Description']=='Adjust bad debt').show()

26/04/19 12:29:38 WARN TaskSetManager: Stage 27 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 12:29:38 WARN TaskSetManager: Stage 28 contains a task of very large size (2082 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 12:29:38 WARN TaskSetManager: Stage 29 contains a task of very large size (2033 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+
|Invoice|StockCode|    Description|Quantity|        InvoiceDate|    Price|Customer ID|       Country|
+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+
|A506401|        B|Adjust bad debt|       1|2010-04-29 13:36:00|-53594.36|        NaN|United Kingdom|
|A516228|        B|Adjust bad debt|       1|2010-07-19 11:24:00|-44031.79|        NaN|United Kingdom|
|A528059|        B|Adjust bad debt|       1|2010-10-20 12:04:00|-38925.87|        NaN|United Kingdom|
+-------+---------+---------------+--------+-------------------+---------+-----------+--------------+



In [37]:
df.groupBy('Customer ID').count().orderBy("count",ascending = False).show()

26/04/19 12:32:04 WARN TaskSetManager: Stage 30 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
[Stage 30:>                                                       (0 + 16) / 16]

+-----------+------+
|Customer ID| count|
+-----------+------+
|        NaN|107927|
|    14911.0|  5710|
|    17841.0|  5114|
|    14606.0|  3927|
|    14156.0|  2710|
|    12748.0|  2665|
|    17850.0|  2515|
|    16549.0|  2274|
|    15311.0|  2226|
|    14527.0|  1826|
|    14646.0|  1805|
|    16782.0|  1703|
|    13089.0|  1581|
|    15005.0|  1388|
|    17377.0|  1377|
|    13081.0|  1369|
|    15039.0|  1302|
|    13564.0|  1233|
|    14298.0|  1228|
|    15768.0|  1213|
+-----------+------+
only showing top 20 rows

